# Ollama Demo

### Importing Libraries

This notebook demonstrates how we will use Ollama to answer questions with context from Qdrant. We begin by importing the neccesary libraires

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance, QueryResponse
from ollama import Client as OllamaClient, ChatResponse
from typing import Mapping, Any, Optional, Sequence
import os

### Client Setup

Next we define functions to get Ollama and Qdrant clients

In [4]:
_ollama_client: Optional["OllamaClient"] = None

def _get_ollama_client() -> OllamaClient:
    global _ollama_client
    if _ollama_client is None:
        host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        _ollama_client = OllamaClient(host=host)
    return _ollama_client

In [5]:
_qdrant_client: Optional["QdrantClient"] = None

def _get_qdrant_client() -> QdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        url = os.getenv("QDRANT_HOST", "http://localhost:6333")
        _qdrant_client = QdrantClient(url=url)
    return _qdrant_client

### RAG Query

Next, we define a family of function to run a "RAG" query against a pre-existing Qdrant collection

In [6]:
def embed(text:str) -> Sequence[float]:
    _ollama_client = _get_ollama_client()
    response: ChatResponse = _ollama_client.embed(model="nomic-embed-text", input=text) # type: ignore
    return response.embeddings[0] # type: ignore

In [8]:
def search(vector: Sequence[float]) -> QueryResponse:
    qdrant_client = _get_qdrant_client()
    results: QueryResponse = qdrant_client.query_points(
        collection_name="test_collection",
        query=vector,
        limit=1,
    )
    return results

In [24]:
system_prompt = """You are an AI assistant that helps people find information.
Use the provided context to answer the question."""

system_prompt

'You are an AI assistant that helps people find information.\nUse the provided context to answer the question.'

In [26]:
def build_prompt(user_input: str, context: Sequence[str], system_prompt: str) -> str:
    context_block = "\n".join(
        f"[{idx+1}] {ctx}"
        for idx, ctx in enumerate(context)
    )

    prompt = f"""SYSTEM:
{system_prompt}

CONTEXT:
{context_block}

QUESTION:
{user_input}

ANSWER:
"""
    return prompt


build_prompt("Where is France and what is its capital?", ["France is a country in Europe", "Its capital is Paris."], system_prompt)

'SYSTEM:\nYou are an AI assistant that helps people find information.\nUse the provided context to answer the question.\n\nCONTEXT:\n[1] France is a country in Europe\n[2] Its capital is Paris.\n\nQUESTION:\nWhere is France and what is its capital?\n\nANSWER:\n'

In [9]:
def query(messages: Sequence[Mapping[str, Any]], model: str = "llama3.2") -> ChatResponse:
    """
    messages: sequence of dicts or Message objects, e.g. {"role": "user", "content": "Hello"}
    Returns a ChatResponse (stream=False).
    """
    # Basic structural validation
    for m in messages:
        if isinstance(m, dict):
            if "role" not in m or "content" not in m:
                raise ValueError("message dict must contain 'role' and 'content' keys")

    ollama_client = _get_ollama_client()
    response: ChatResponse = ollama_client.chat(model="llama3.2", messages=messages, stream=False) # type: ignore

    return response